# DataWorks 2026 workflow

This is the single project notebook. Run cells from top to bottom. It never edits the original workbooks in `TestCases/`.

## Setup

Choose the `.venv` Python kernel in VS Code before running this notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'TestCases').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

INPUT_DIR = ROOT / 'TestCases'
PROCESSED_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR = ROOT / 'outputs'

from src.stage1_load_core import run as run_core
from src.stage1_clean_curves import run as run_curves
from src.stage2_history_matching import run as run_history_matching
from src.stage3_baseline_forecast import run as run_baseline_forecast
from src.stage3_forecast_backtest import run_backtest


## Stage 1A — core data loading and cleaning

Loads field history and the 100 uncertainty cases. The visual uses raw source data before cleaning.

In [ ]:
core_audit = run_core(INPUT_DIR, PROCESSED_DIR, OUTPUT_DIR)
display(core_audit)
display(Image(filename=OUTPUT_DIR / 'core_data_before_cleaning.png'))

## Stage 1B — simulated production curves

Converts the six case-curve sheets to tidy tables and aligns historical observations to quarterly dates. The supplied curves stop on 1 Jan 2008, so future template dates remain unavailable rather than being extrapolated.

In [ ]:
curve_audit = run_curves(INPUT_DIR, PROCESSED_DIR, OUTPUT_DIR)
display(curve_audit)
display(Image(filename=OUTPUT_DIR / 'production_curves_before_quarterly_resampling.png'))

## Stage 2 — baseline history matching

Ranks all 100 cases using weighted normalized RMSE between simulated and observed historical oil, gas, and water rates. A lower score is a better match. This is a transparent baseline, not the final 2008–2028 forecast model.

In [ ]:
ranking, score_components = run_history_matching(PROCESSED_DIR, OUTPUT_DIR)
display(ranking.head(10))
display(score_components.head(12))
display(Image(filename=OUTPUT_DIR / 'top_history_matches.png'))

## Stage 1C — grid manifest and schema validation

Checks all 100 large grid sheets without loading every grid cell into memory. It records file, case, shape, and schema so the ML stage can use the files safely.

In [ ]:
import re
from openpyxl import load_workbook

EXPECTED_GRID_HEADERS = [
    'I Index', 'J Index', 'K Index', 'X Coordinate', 'Y Coordinate', 'Z Coordinate',
    'Pore Volume (PORV)', 'Porosity (PORO)', 'Permeability I (PERMX)',
    'Permeability J (PERMY)', 'Permeability K (PERMZ)', 'Net-to-Gross Ratio (NTG)',
    'Transmissibility I (TRANX)', 'Transmissibility J (TRANY)', 'Transmissibility K (TRANZ)',
    'Fluid In Place Region (FIPNUM)', 'PVT Region (PVTNUM)',
    'Connate Water Saturation (SWL)', 'Critical Water Saturation (SWCR)',
    'Maximum Water Saturation (SWU)', 'Connateg Gas Saturation (SGL)',
    'Critical Gas Saturation (SGCR)', 'Residual Oil to Water (SOWCR)',
    'Pressure (PRESSURE)', 'Water Saturation (SWAT)', 'Oil Saturation (SOIL)',
    'Gas Saturation (SGAS)', 'Solution Gas-Oil Ratio (RS)', 'Vapor Oil-Gas Ratio (RV)'
]

grid_files = sorted((INPUT_DIR).glob('0[3-8]*.xlsx'))
manifest_records = []
for path in grid_files:
    workbook = load_workbook(path, read_only=True, data_only=True)
    for sheet in workbook.sheetnames:
        worksheet = workbook[sheet]
        case_id = int(re.fullmatch(r'Case (\d+)', sheet).group(1))
        raw_headers = [cell.value for cell in next(worksheet.iter_rows(min_row=1, max_row=1))]
        normalised_headers = [re.sub(r'\s+', ' ', str(header)).strip() for header in raw_headers]
        manifest_records.append({
            'source_file': path.name,
            'sheet_name': sheet,
            'case_id': case_id,
            'data_rows': worksheet.max_row - 1,
            'columns': worksheet.max_column,
            'schema_valid': normalised_headers == EXPECTED_GRID_HEADERS,
            'saturation_column_raw_name': raw_headers[20],
        })
    workbook.close()

grid_manifest = pd.DataFrame(manifest_records).sort_values('case_id').reset_index(drop=True)
grid_audit = pd.DataFrame([
    {'check': 'case_sheets', 'observed': len(grid_manifest), 'expected': 100},
    {'check': 'case_ids_1_to_100', 'observed': set(grid_manifest.case_id) == set(range(1, 101)), 'expected': True},
    {'check': 'data_rows_per_case', 'observed': grid_manifest.data_rows.nunique(), 'expected': 1},
    {'check': 'expected_data_rows', 'observed': int(grid_manifest.data_rows.iloc[0]), 'expected': 90365},
    {'check': 'expected_columns', 'observed': int(grid_manifest['columns'].iloc[0]), 'expected': 29},
    {'check': 'valid_schemas', 'observed': int(grid_manifest.schema_valid.sum()), 'expected': 100},
])
grid_audit['status'] = grid_audit.apply(lambda row: 'pass' if str(row.observed) == str(row.expected) else 'flag', axis=1)

grid_manifest.to_csv(PROCESSED_DIR / 'grid_manifest.csv', index=False)
grid_audit.to_csv(PROCESSED_DIR / 'grid_schema_audit.csv', index=False)
display(grid_audit)
display(grid_manifest.head())

## Stage 3A — forecast back-test

The supplied production curves stop on 1 Jan 2008, so future values must be modelled. To choose conservative assumptions honestly, this test hides 2004–2008 data, forecasts that known period from earlier data, and compares three parameter sets. Lower weighted normalized RMSE is better.

In [ ]:
backtest_results, backtest_summary = run_backtest(PROCESSED_DIR, OUTPUT_DIR)
display(backtest_summary)
display(Image(filename=OUTPUT_DIR / 'forecast_backtest_comparison.png'))


## Stage 3B — future baseline forecast

This forecast automatically uses the best back-tested configuration. It projects gas and oil with bounded exponential decline and water towards a bounded plateau. It remains a transparent benchmark for Person 3, not a final competition submission.

In [ ]:
forecast_output, forecast_parameters, forecast_audit = run_baseline_forecast(
    INPUT_DIR, PROCESSED_DIR, OUTPUT_DIR
)
display(forecast_parameters)
display(forecast_audit)
display(Image(filename=OUTPUT_DIR / 'baseline_forecast_ensemble.png'))


## Handover tables

Person 2 reviews the ranking and score components. Person 3 starts from the baseline forecast parameters and audit, then evaluates any improvement without changing the data split or using test cases for tuning.

In [ ]:
field_history = pd.read_csv(PROCESSED_DIR / 'field_history.csv', parse_dates=['date'])
case_curves = pd.read_csv(PROCESSED_DIR / 'case_production_quarterly.csv', parse_dates=['date'])
coverage = pd.read_csv(PROCESSED_DIR / 'case_production_quarterly_coverage.csv')

display(field_history.head())
display(case_curves.head())
display(coverage)